# 04 — Flat-Foldability and the Folded State

A crease pattern is **flat-foldable** if it can be folded onto a single plane without self-intersections. Two classical local necessary conditions are:

- **Kawasaki**: at every interior vertex, the alternating sum of consecutive sector angles is zero.
- **Maekawa**: at every interior vertex, |#mountains − #valleys| = 2.

These are necessary but not sufficient — globally one must also pick a consistent face stacking order. Eucare solves the global problem as an integer linear program (`overlap.fold_complete`).

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    reciprocal_figures,
    rendering,
)
from eucare.rendering import multi_show


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


## Build a CP and check Kawasaki at every interior vertex

In [ ]:
from eucare.reciprocal_figures import kawasaki_sum

G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)

interior = [v for v in SRG.vertices if not v.on_border()]
residuals = [abs(kawasaki_sum(v)) for v in interior]
print(f'{len(interior)} interior vertices')
print(f'max |Kawasaki residual| = {max(residuals):.2e}')


## Solve the folded face order

`overlap.fold_complete` writes mountain/valley crease assignments into the CP and returns a dict with the CP plus top/bottom views of the folded state. It uses the PuLP modeller; CBC is the default solver, CPLEX is used automatically if available.

We set `TQDM_DISABLE` in the very first cell of every tutorial (see the setup cell) so the progress bars don't flood the notebook output. To run with progress bars in a normal session, simply remove that line.

In [ ]:
result = overlap.fold_complete(SRG, overlap_eps=1e-8)
CP = result['CP']
n_m = sum(1 for h in CP.halfedges_representing_edges()
          if h.attributes.get(overlap.CREASE_ASSIGNMENT) == overlap.MOUNTAIN)
n_v = sum(1 for h in CP.halfedges_representing_edges()
          if h.attributes.get(overlap.CREASE_ASSIGNMENT) == overlap.VALLEY)
print(f'mountain creases: {n_m}, valley creases: {n_v}')


## Visualise the M/V assignment

After `fold_complete` the CP already carries `color_key` attributes for every crease (red mountain / blue valley), so a plain `CP.show(**CREASE_PATTERN_PRESET)` is enough.

In [ ]:
CP.show(**rendering.CREASE_PATTERN_PRESET)


## Top and bottom of the folded state

`result['folded_view_top']` and `result['folded_view_bottom']` are views of the same folded shape from above and below. Stacked faces inherit colour keys from their order, so showing both views side-by-side is the quickest way to see the folded geometry.

In [ ]:
multi_show(
    [result['folded_view_top'], result['folded_view_bottom']],
    titles=['folded — top', 'folded — bottom'],
    face_inset=0.0, render_vertices=False, render_faces=True,
)


## Saving everything to disk

`overlap.save_results` packages the CP, both folded views, a back-lit composite, and a plotter-ready SVG into a single output directory.

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    overlap.save_results(result, path=d, bbox=(20, 20))
    print('produced files:')
    for name in sorted(os.listdir(d)):
        print(' ', name)


## What's next

- [`05_Conway_Plus_SRG`](05_Conway_Plus_SRG.ipynb) — design more interesting CPs by composing Conway operators before SRG.
- [`06_Export_and_3D`](06_Export_and_3D.ipynb) — save the result to SVG / FOLD / STL.
- [`07_Styling`](07_Styling.ipynb) — custom colours and edge widths.